# Deploy Trained Smolvla Policy

<img src="./media/rollout3.gif" width="480" height="360">

Deploy trained policy in simulation.

In [11]:
# !pip install transformers==4.50.3
# !pip install num2words
# !pip install accelerate
# !pip install safetensors>=0.4.3

### [Optional] Download Dataset

In [12]:
# '''
# If you want to use the collected dataset, please download it from Hugging Face.
# '''
# !git clone https://huggingface.co/datasets/Jeongeun/omy_pnp_language

## Step 2. Train Model

In [13]:
# !python train_model.py --config_path smolvla_omy.yaml

## Step 3. Deploy

In [14]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
import numpy as np
from lerobot.common.datasets.utils import write_json, serialize_dict
from lerobot.common.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.common.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.configs.types import FeatureType
from lerobot.common.datasets.factory import resolve_delta_timestamps
from lerobot.common.datasets.utils import dataset_to_policy_features
import torch
from PIL import Image
import torchvision

In [15]:
device = 'cuda'

In [16]:
try:
    dataset_metadata = LeRobotDatasetMetadata("omy_pnp_language", root='./demo_data_language')
except:
    dataset_metadata = LeRobotDatasetMetadata("omy_pnp_language", root='./omy_pnp_language')
features = dataset_to_policy_features(dataset_metadata.features)
output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}
# Policies are initialized with a configuration class, in this case `DiffusionConfig`. For this example,
# we'll just use the defaults and so no arguments other than input/output features need to be passed.
# Temporal ensemble to make smoother trajectory predictions
cfg = SmolVLAConfig(input_features=input_features, output_features=output_features, chunk_size= 5, n_action_steps=5)
delta_timestamps = resolve_delta_timestamps(cfg, dataset_metadata)

In [17]:
# We can now instantiate our policy with this config and the dataset stats.
policy = SmolVLAPolicy.from_pretrained('./ckpt/smolvla_omy/checkpoints/last/pretrained_model', dataset_stats=dataset_metadata.stats)
# You can load the trained policy from hub if you don't have the resources to train it.
# policy = SmolVLAPolicy.from_pretrained("Jeongeun/omy_pnp_pi0", config=cfg, dataset_stats=dataset_metadata.stats)
policy.to(device)



'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: de48e9ce-877b-4597-be1d-7d1c9c727b0f)')' thrown while requesting HEAD https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


Reducing the number of VLM layers to 16 ...
Loading weights from local directory


SmolVLAPolicy(
  (normalize_inputs): Normalize(
    (buffer_observation_state): ParameterDict(
        (mean): Parameter containing: [torch.cuda.FloatTensor of size 6 (cuda:0)]
        (std): Parameter containing: [torch.cuda.FloatTensor of size 6 (cuda:0)]
    )
  )
  (normalize_targets): Normalize(
    (buffer_action): ParameterDict(
        (mean): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
        (std): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
    )
  )
  (unnormalize_outputs): Unnormalize(
    (buffer_action): ParameterDict(
        (mean): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
        (std): Parameter containing: [torch.cuda.FloatTensor of size 7 (cuda:0)]
    )
  )
  (model): VLAFlowMatching(
    (vlm_with_expert): SmolVLMWithExpertModel(
      (vlm): SmolVLMForConditionalGeneration(
        (model): SmolVLMModel(
          (vision_model): SmolVLMVisionTransformer(
            (embeddings): SmolVLMVisio

In [18]:
from mujoco_env.y_env2 import SimpleEnv2
xml_path = './asset/example_scene_y2.xml'
PnPEnv = SimpleEnv2(xml_path, action_type='joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[31] n_qvel:[28] n_qacc:[28] n_ctrl:[10]
 integrator:[IMPLICITFAST]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] [link1] mass:[2.06]kg
 [6/23] [link2] mass:[3.68]kg
 [7/23] [link3] mass:[2.39]kg
 [8/23] [link4] mass:[1.40]kg
 [9/23] [link5] mass:[1.40]kg
 [10/23] [link6] mass:[0.65]kg
 [11/23] [camera_center] mass:[0.00]kg
 [12/23] [tcp_link] mass:[0.32]kg
 [13/23] [rh_p12_rn_r1] mass:[0.07]kg
 [14/23] [rh_p12_rn_r2] mass:[0.02]kg
 [15/23] [rh_p12_rn_l1] mass:[0.07]kg
 [16/23] [rh_p12_rn_l2] mass:[0.02]kg
 [17/23] [body_obj_mug_5] mass:[0.00]kg
 [18/23] [object_mug_5] mass:[0.08]kg
 [19/23] [body_obj_plate_11] mass:[0.00]kg
 [20/23] [object_plate_11] mass:[0.10]kg
 [21/23] [body_obj_mug_6] mass:[0.00]kg
 [22/23] [object_mug

In [19]:
from torchvision import transforms
# Approach 1: Using torchvision.transforms
def get_default_transform(image_size: int = 224):
    """
    Returns a torchvision transform that:
     Converts to a FloatTensor and scales pixel values [0,255] -> [0.0,1.0]
    """
    return transforms.Compose([
        transforms.ToTensor(),  # PIL [0–255] -> FloatTensor [0.0–1.0], shape C×H×W
    ])

In [20]:
NUM_EVAL_EPISODES = 50
MAX_STEPS_PER_EPISODE = 400
EVAL_TASKS = (
    'Place the red mug on the plate.',
    'Place the blue mug on the plate.',
)

episode_count = 0
success_count = 0
step = 0

PnPEnv.reset(seed=0)
PnPEnv.set_instruction(EVAL_TASKS[episode_count % len(EVAL_TASKS)])
policy.reset()
policy.eval()
IMG_TRANSFORM = get_default_transform()

while PnPEnv.env.is_viewer_alive() and episode_count < NUM_EVAL_EPISODES:
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # Success is checked after the previous command has been simulated.
        success = PnPEnv.check_success()
        timed_out = step >= MAX_STEPS_PER_EPISODE
        if success or timed_out:
            success_count += int(success)
            episode_count += 1
            result = 'SUCCESS' if success else 'FAIL (timeout)'
            success_rate = 100.0 * success_count / episode_count
            print(
                f'Episode {episode_count}/{NUM_EVAL_EPISODES}: {result} | '
                f'Success rate: {success_count}/{episode_count} ({success_rate:.1f}%)'
            )

            if episode_count >= NUM_EVAL_EPISODES:
                break

            # Start the next episode and clear the policy action queue.
            PnPEnv.reset()
            PnPEnv.set_instruction(EVAL_TASKS[episode_count % len(EVAL_TASKS)])
            policy.reset()
            step = 0
            continue

        # Get the current state of the environment
        state = PnPEnv.get_joint_state()[:6]
        # Get the current image from the environment
        image, wirst_image = PnPEnv.grab_image()
        image = Image.fromarray(image)
        image = image.resize((256, 256))
        image = IMG_TRANSFORM(image)
        wrist_image = Image.fromarray(wirst_image)
        wrist_image = wrist_image.resize((256, 256))
        wrist_image = IMG_TRANSFORM(wrist_image)
        data = {
            'observation.state': torch.from_numpy(state).unsqueeze(0).to(device),
            'observation.image': image.unsqueeze(0).to(device),
            'observation.wrist_image': wrist_image.unsqueeze(0).to(device),
            'task': [PnPEnv.instruction],
        }
        # Select an action
        with torch.inference_mode():
            action = policy.select_action(data)
        action = action[0,:7].cpu().detach().numpy()
        # Take a step in the environment
        _ = PnPEnv.step(action)
        PnPEnv.render(idx=episode_count + 1)
        step += 1

if episode_count > 0:
    final_rate = 100.0 * success_count / episode_count
    print(f'Final success rate: {success_count}/{episode_count} ({final_rate:.1f}%)')
else:
    print('Evaluation stopped before any episode finished.')

DONE INITIALIZATION
Episode 1/50: SUCCESS | Success rate: 1/1 (100.0%)
DONE INITIALIZATION
Episode 2/50: SUCCESS | Success rate: 2/2 (100.0%)
DONE INITIALIZATION
Episode 3/50: SUCCESS | Success rate: 3/3 (100.0%)
DONE INITIALIZATION
Episode 4/50: SUCCESS | Success rate: 4/4 (100.0%)
DONE INITIALIZATION
Episode 5/50: SUCCESS | Success rate: 5/5 (100.0%)
DONE INITIALIZATION
Episode 6/50: FAIL (timeout) | Success rate: 5/6 (83.3%)
DONE INITIALIZATION
Episode 7/50: SUCCESS | Success rate: 6/7 (85.7%)
DONE INITIALIZATION
Episode 8/50: FAIL (timeout) | Success rate: 6/8 (75.0%)
DONE INITIALIZATION
Episode 9/50: SUCCESS | Success rate: 7/9 (77.8%)
DONE INITIALIZATION
Episode 10/50: SUCCESS | Success rate: 8/10 (80.0%)
DONE INITIALIZATION
Episode 11/50: FAIL (timeout) | Success rate: 8/11 (72.7%)
DONE INITIALIZATION
Episode 12/50: SUCCESS | Success rate: 9/12 (75.0%)
DONE INITIALIZATION
Episode 13/50: SUCCESS | Success rate: 10/13 (76.9%)
DONE INITIALIZATION
Episode 14/50: FAIL (timeout) | Suc